# 🗜️ Memory Optimizations: Keeping Conversations Affordable

The previous notebook gave our agent perfect recall by replaying the entire conversation on
every turn. That works — right up until it doesn't. Every extra token costs money and adds
latency, and both grow **linearly with the length of the conversation**.

This notebook builds four memory strategies on the same customer-support scenario and
compares what each one costs.

## Learning Objectives
In this notebook, you will learn:
1. **Sequential memory** - the full-history baseline, and exactly how it degrades
2. **Sliding window memory** - deleting old messages with `RemoveMessage`, and what you lose
3. **Trim on read** - sending the model a window with `trim_messages` while the thread keeps everything
4. **Summarization memory** - compressing old turns instead of discarding them
5. **Measuring the trade-off** - prompt tokens per turn for all four strategies, side by side

## Where this fits
`01_memory_and_conversational_agent.ipynb` stored every message on a thread and replayed all of it on every turn.
Module 01's `05_trimming_windowing_and_summary.ipynb` bounded an LCEL chat history with the same three ideas —
windows, trimming and running summaries. This notebook does it inside a LangGraph graph, where the question that
matters is whether old messages are *deleted* from the checkpointed state or only *hidden* from the model.

## Prerequisites
- **`01_memory_and_conversational_agent.ipynb`** — checkpointers and threads
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

### Key Concepts:
- **Reducer**: the function that merges a node's return value into state (`add` vs `add_messages`)
- **`RemoveMessage`**: a sentinel that tells `add_messages` to *delete* a message by id
- **`trim_messages`**: picks the newest messages (or tokens) of a list for one call — it changes nothing in state
- **Context window**: the token budget one model call must fit inside

> **The trade-off in one line**: sequential never forgets but never stops growing; sliding
> window is cheap but deletes old turns for good; trim on read sends the same small prompt but keeps
> the full thread in storage; summarization keeps the gist at the cost of an extra LLM call per fold.

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` from the project-root `.env`; the next cell creates the one chat model every
strategy below shares, and the one after that defines the per-turn instrumentation.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import time
from operator import add
from typing import Annotated

from langchain_core.messages import AnyMessage, HumanMessage, RemoveMessage, SystemMessage, trim_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by all four memory strategies
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

In [ ]:
# ============================================================================
# INSTRUMENTATION: Measure Every Turn
# ============================================================================
# Each strategy is judged on the same numbers, so we time the call and read token
# usage off the reply in one place. Prompt (input) tokens are what memory adds to
# every call; TOKEN_LOG keeps them per thread for the comparison tables.
TOKEN_LOG = {}  # thread_id -> prompt tokens of each answering call, in turn order


def timed_invoke(graph, state: dict, config: dict, channel: str = "messages", show: bool = True):
    """Invoke a graph, print the exchange, and report latency + token usage."""
    started = time.time()
    response = graph.invoke(state, config=config)
    latency = time.time() - started

    ai_msg = response[channel][-1]
    usage = ai_msg.usage_metadata  # ChatOpenAI fills this in on every reply
    TOKEN_LOG.setdefault(config["configurable"]["thread_id"], []).append(usage["input_tokens"])

    if show:
        state[channel][-1].pretty_print()
        ai_msg.pretty_print()
        # ANSI: cyan for latency, green for tokens
        print(f"\n\033[96m⏱️  Latency: {latency:.2f}s\033[0m")
        print(f"\033[92m📋 Prompt tokens: {usage['input_tokens']} · total: {usage['total_tokens']}\033[0m")

    return response


def run_conversation(graph, turns: list[str], thread_id: str, channel: str = "messages", show: bool = True):
    """Send each user turn, in order, on one thread."""
    TOKEN_LOG[thread_id] = []
    config = {"configurable": {"thread_id": thread_id}}
    for n, text in enumerate(turns, 1):
        if show:
            print(f"\n{'=' * 34} Turn {n} {'=' * 34}")
        timed_invoke(graph, {channel: [HumanMessage(content=text)]}, config, channel, show)


def print_token_table(threads: dict) -> None:
    """Prompt tokens per turn, one column per strategy: threads = {label: thread_id}."""
    print("turn" + "".join(f"{label:>16}" for label in threads))
    for n, row in enumerate(zip(*(TOKEN_LOG[t] for t in threads.values())), 1):
        print(f"{n:>4}" + "".join(f"{tokens:>16}" for tokens in row))


print("✅ Instrumentation ready!")


---
## 📜 Part 1: Sequential Memory (The Baseline)

The simplest possible policy: **keep everything**. Every turn is appended to state, and the
whole list is sent to the model on every call.

Note the reducer here is `operator.add` — plain list concatenation. It appends and never
removes, which is exactly the behaviour we want to measure before improving on it.

In [ ]:
# ============================================================================
# STATE SCHEMA: Sequential (Append-Only)
# ============================================================================
# `add` is plain list concatenation - nothing is ever dropped.

class SequentialState(TypedDict):
    messages: Annotated[list[AnyMessage], add]


print("✅ Sequential state schema defined!")

In [ ]:
# ============================================================================
# CHAT NODE: Send the Entire History
# ============================================================================
# Short answers keep six turns readable and cheap; every strategy below uses this prompt.
SUPPORT_PROMPT = SystemMessage(content="You are a customer support assistant. Keep every answer under 60 words.")


def chat_llm_node(state: SequentialState):
    """Prepend the system prompt, then send every message we have."""
    history = [SUPPORT_PROMPT]
    history.extend(state["messages"])

    reply = llm.invoke(history)

    return {"messages": [reply]}


print("✅ Sequential chat node defined!")


In [ ]:
# ============================================================================
# GRAPH: Sequential Memory
# ============================================================================
# flow: START -> chat_llm -> END

checkpointer = InMemorySaver()

builder = StateGraph(SequentialState)
builder.add_node("chat_llm", chat_llm_node)
builder.add_edge(START, "chat_llm")
builder.add_edge("chat_llm", END)

graph = builder.compile(checkpointer=checkpointer)

print("✅ Sequential graph compiled!")

### 1.1 🧪 Six Turns, Measured

Every strategy in this notebook gets the same six-turn support conversation, so their numbers line up.
Turn 1 slips in an extra detail — *"My favorite color is blue by the way."* — and turn 3 asks for it back.

Watch the prompt-token count on each turn. The point is the *trend*, not the answer quality.

In [ ]:
# ============================================================================
# THE TEST CONVERSATION: Six User Turns, Shared by Every Strategy
# ============================================================================
SUPPORT_TURNS = [
    "Hi, I'm being charged twice for my subscription. Can you help me figure out what's going on? "
    "My favorite color is blue by the way.",
    "I think this started after I changed my billing address last month.",
    "What is my favorite color?",                                   # needs turn 1
    "Both charges are $29.99 and they land on the 3rd of every month.",
    "Can you refund the duplicate charge to my card?",
    "Thanks. What should I check so this does not happen again?",
]

for n, text in enumerate(SUPPORT_TURNS, 1):
    print(f"{n}. {text}")


In [ ]:
# ============================================================================
# SEQUENTIAL: Six Turns on One Thread
# ============================================================================
# Turn 3 can be answered: turn 1 is still part of every prompt.
run_conversation(graph, SUPPORT_TURNS, "ticket-seq")


### 1.2 📊 What the Numbers Show

**Prompt tokens grow roughly linearly with the number of turns.** You pay for the entire
past conversation on every single call — which is also why turn 3 can name the colour from turn 1.
On long threads, response time climbs with prompt size too.

For short conversations this is fine — no detail is ever lost, and for internal tools or
early prototypes that may be all you need. But it does not scale:

- **Every extra token has a cost.** An unmanaged history becomes financially unsustainable.
- **Large prompts mean high latency.** Slow answers are unacceptable in production.

Sequential memory is best treated as a **baseline**: the most straightforward behaviour,
easy to instrument, and the thing every other strategy is measured against.

The next step keeps the spirit of it — don't lose what matters — while putting a hard
ceiling on context.

---
## 🪟 Part 2: Sliding Window Memory

Instead of retaining everything, keep only the **most recent N messages**. As new messages
arrive, the oldest drop off and the window slides forward.

The flow becomes:

1. The user and agent take turns; each turn is appended to the thread's state
2. Before generating a reply, a `truncate` node deletes every message older than the last N
3. The remaining window — which ends with the new question — goes to the LLM
4. Everything outside the window is **deleted from the checkpointed state**, not just hidden for this turn:
   it is gone for every later turn, for `get_state()`, and for anyone auditing the thread

Think of it as a short-term memory buffer: the agent remembers what just happened, but
starts forgetting the earliest parts of the conversation.

### Key Concepts:
- **`add_messages` reducer**: unlike `add`, it understands message *ids* — which is what
  makes deletion possible
- **`RemoveMessage(id=...)`**: returning one tells the reducer to delete that message from state

In [ ]:
# ============================================================================
# STATE SCHEMA: Sliding Window
# ============================================================================
# Note the reducer change: add_messages (not add). Only add_messages knows how
# to interpret RemoveMessage, which is how we drop old turns. (This is exactly the
# schema LangGraph ships as langgraph.graph.MessagesState.)

class WindowState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


print("✅ Sliding-window state schema defined!")

In [ ]:
# ============================================================================
# TRUNCATION NODE: Enforce the Window
# ============================================================================

# 3 = the previous exchange (user + assistant) plus the new question. The whole first
# exchange must leave the window: if the assistant's first reply stayed, it could
# repeat the detail we test for and hide the forgetting.
MAX_MESSAGES = 3


def truncate_messages_node(state: WindowState):
    """Drop everything older than the last MAX_MESSAGES messages."""
    msgs = state["messages"]
    print(f"🪟 [truncate] incoming length = {len(msgs)}")

    if len(msgs) <= MAX_MESSAGES:
        print("🪟 [truncate] no-op (under limit)")
        return {}

    to_drop = msgs[:-MAX_MESSAGES]
    print(f"🗑️  [truncate] dropping {len(to_drop)} message(s)")

    # RemoveMessage is a deletion instruction, not a message the model will see.
    return {"messages": [RemoveMessage(id=m.id) for m in to_drop]}


print("✅ Truncation node defined!")

In [ ]:
# ============================================================================
# CHAT NODE: Instrumented to Show What Survived the Window
# ============================================================================

def chat_llm_node(state: WindowState):
    """Same as before, but prints what the model actually receives."""
    history = [SUPPORT_PROMPT]

    print(f"🤖 [chat_llm] LLM sees {len(state['messages'])} message(s):")
    for m in state["messages"]:
        print(f"     {m.type}: {m.content[:70]}")

    history.extend(state["messages"])

    reply = llm.invoke(history)

    return {"messages": [reply]}


print("✅ Instrumented chat node defined!")


In [ ]:
# ============================================================================
# GRAPH: Sliding Window Memory
# ============================================================================
# flow: START -> truncate -> chat_llm -> END
# Truncation runs BEFORE the model call, so the window is enforced every turn.

checkpointer = InMemorySaver()

builder = StateGraph(WindowState)
builder.add_node("truncate", truncate_messages_node)
builder.add_node("chat_llm", chat_llm_node)

builder.add_edge(START, "truncate")
builder.add_edge("truncate", "chat_llm")
builder.add_edge("chat_llm", END)

sliding_graph = builder.compile(checkpointer=checkpointer)

print("✅ Sliding-window graph compiled!")

### 2.1 🧪 The Same Conversation, Windowed

The same six turns. By turn 3 the whole first exchange — the question *and* the assistant's reply, which may
have echoed the colour — has been deleted from the thread, and the agent genuinely cannot answer.

That failure is the lesson, not a bug.

In [ ]:
# ============================================================================
# SLIDING WINDOW: Six Turns on One Thread
# ============================================================================
# Watch the [truncate] lines: from turn 3 on, every turn deletes the oldest exchange,
# so turn 3 ("What is my favorite color?") has no way to see turn 1.
run_conversation(sliding_graph, SUPPORT_TURNS, "ticket-sliding")


In [ ]:
# ============================================================================
# COMPARE: Prompt Tokens per Turn, Sequential vs Sliding Window
# ============================================================================
print_token_table({"sequential": "ticket-seq", "sliding window": "ticket-sliding"})


### 2.2 📊 What the Numbers Show

**Prompt tokens per turn grow only until the window fills, then flatten.** In the table, the
sliding-window column levels off from turn 3 on, while the sequential column keeps climbing with every
turn. That is the whole win: the prompt — and on long threads the latency — stops growing.

The cost is equally clear: **anything outside the window is gone.** For a support agent that
bites when:

- A critical fact was mentioned early (*"I'm on the Enterprise plan"*, *"only happens on Firefox"*)
  and has since scrolled away
- The user refers back to something many turns ago
- Tickets run long and interleave several threads of discussion

Sliding window keeps cost and latency under control, and works well for short, focused
conversations where what matters is recent. It has no way to remember older details.

Before fixing that with summarization, look at a second way to bound the prompt — one that does not delete anything.

---
## ✂️ Part 3: Trim on Read

Part 2 bounded the prompt by **deleting** old messages from state. There is a second way: leave the stored
thread alone and choose what to *send*. The checkpointer keeps the complete conversation — which is what makes
time-travel, resume and auditing possible — and the model node cuts a window out of it with `trim_messages`
on every call.

The window has the same size as Part 2's (`MAX_MESSAGES` messages), so the model gets the same prompt; only
storage differs. And because nothing is ever deleted, the append-only `SequentialState` from Part 1 is all the
state this needs — no reducer change.

### Key Concepts:
- **`trim_messages(..., strategy="last", max_tokens=N)`**: keeps the newest messages that fit the budget
- **`token_counter=len`**: counts *messages* instead of tokens, to mirror Part 2's window. In production, pass a
  real counter (the model, or `count_tokens_approximately`) and a token budget
- **`start_on="human"`**: never start the window on a dangling assistant reply

In [ ]:
# ============================================================================
# CHAT NODE + GRAPH: Trim What the Model Sees, Store Everything
# ============================================================================
# flow: START -> chat_llm -> END   (no truncate node: nothing is ever deleted)

def chat_llm_trimmed_node(state: SequentialState):
    """Send only the newest MAX_MESSAGES messages; the stored history is left untouched."""
    window = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=len,         # count messages, not tokens, to mirror Part 2's window
        max_tokens=MAX_MESSAGES,
        start_on="human",          # never start the window on an assistant reply
    )
    print(f"✂️  [chat_llm] thread stores {len(state['messages'])} message(s), LLM sees {len(window)}:")
    for m in window:
        print(f"     {m.type}: {m.content[:70]}")

    reply = llm.invoke([SUPPORT_PROMPT] + window)

    return {"messages": [reply]}


builder = StateGraph(SequentialState)
builder.add_node("chat_llm", chat_llm_trimmed_node)
builder.add_edge(START, "chat_llm")
builder.add_edge("chat_llm", END)

trim_graph = builder.compile(checkpointer=InMemorySaver())

print("✅ Trim-on-read graph compiled!")


### 3.1 🧪 The Same Conversation, Trimmed

Turn 3 fails exactly as in Part 2: the model sees the same three messages. Watch the first number on each
`[chat_llm]` line instead — the stored thread keeps growing.

In [ ]:
# ============================================================================
# TRIM ON READ: Six Turns on One Thread
# ============================================================================
run_conversation(trim_graph, SUPPORT_TURNS, "ticket-trim")


### 3.2 🔍 Delete vs View

Same window, same prompt — now compare what each thread still holds, and the prompt tokens of all three
strategies so far.

In [ ]:
# ============================================================================
# DELETE vs VIEW: Same Prompt, Different Stored Thread
# ============================================================================
for label, bounded_graph, thread_id in [
    ("sliding window (delete)", sliding_graph, "ticket-sliding"),
    ("trim on read (view)", trim_graph, "ticket-trim"),
]:
    stored = bounded_graph.get_state({"configurable": {"thread_id": thread_id}}).values["messages"]
    has_colour = any("blue" in m.content for m in stored if m.type == "human")
    print(f"{label:<24} stores {len(stored):>2} messages · turn-1 colour still stored: {has_colour}")

print()
print_token_table({"sequential": "ticket-seq", "window": "ticket-sliding", "trim on read": "ticket-trim"})


Both bounded strategies send the same prompt, so their token columns track each other (answers vary a little
from run to run) and stay flat while the sequential column climbs. Storage is where they part ways: the
sliding-window thread holds only the last four messages (two exchanges), while the trimmed thread still holds all twelve
messages — turn 1's colour included.

| | Sliding window (`RemoveMessage`) | Trim on read (`trim_messages`) |
|---|---|---|
| Where it runs | a `truncate` node that writes deletions to state | inside the model node, on a copy of the list |
| Stored thread | the last N messages — older ones are gone for good | every message, forever |
| What the model sees | the last N messages | the last N messages |
| Change N later, audit, time-travel | ❌ the old turns no longer exist | ✅ the full thread is still there |
| Checkpoint size | bounded | grows with the conversation |

> **Gotcha**: trimming the prompt does not trim the checkpoint. Use `trim_messages` for context-window and
> cost control, not to shrink storage — with `SqliteSaver` the full thread is written to disk on every step.

---
## 🧬 Part 4: Summarization-Based Memory

If you have used an AI coding assistant, you have already seen this. As the context window
fills, instead of failing or blindly truncating, the tool quietly:

- compresses older parts of the conversation into short summaries
- retains key decisions, constraints, and naming conventions
- keeps only the most recent turns in full detail

The assistant still remembers what mattered, while the prompt stays a manageable size.

Instead of dropping old information, we **remember it in condensed form**: periodically
summarise the conversation so far and use that summary in place of the full history.

### Key Concepts:
- **Two channels**: `summary` (a plain string, overwritten each time) and `buffer` (recent messages)
- **Fold, then forget**: once the buffer reaches the threshold, the older messages are summarised *and removed*
  from the buffer with `RemoveMessage`, so the prompt holds the summary plus only the newest message
- **The cost**: one extra LLM call per summarisation — you trade tokens for tokens

In [ ]:
# ============================================================================
# STATE SCHEMA: Summary + Recent Buffer
# ============================================================================
# `summary` has no reducer, so writing it replaces the old value.
# `buffer` uses add_messages, so recent turns accumulate.

class SummarizationState(TypedDict):
    summary: str
    buffer: Annotated[list[AnyMessage], add_messages]


print("✅ Summarization state schema defined!")

In [ ]:
# ============================================================================
# SUMMARIZATION NODE: Compress the Buffer, Then Drop What Was Compressed
# ============================================================================
MESSAGE_THRESHOLD = 4


def summarize_buffer_node(state: SummarizationState):
    """Fold all but the newest message into the running summary and remove them from the buffer."""
    buffer = state["buffer"]
    current_summary = state.get("summary", "")

    if len(buffer) < MESSAGE_THRESHOLD:
        print(f"🧬 [summarize] buffer has {len(buffer)} message(s) — below threshold, no-op")
        return {}

    to_fold = buffer[:-1]  # keep the newest message (the current question) verbatim

    # --- Render the older messages as a transcript for the summarizer ---
    buffer_text = "\n".join(
        f"{'User' if isinstance(m, HumanMessage) else 'Assistant'}: {m.content}" for m in to_fold
    )

    system = SystemMessage(content=(
        "Maintain a summary of a support conversation.\n"
        "Keep only important facts and decisions.\n"
        "Prefer short bullet points when possible."
    ))
    user_content = (
        f"Previous summary:\n{current_summary or '(none)'}\n\n"
        f"New conversation turns:\n{buffer_text}\n\n"
        "Produce an updated summary."
    )
    result = llm.invoke([system, HumanMessage(content=user_content)])
    new_summary = result.content

    print(f"🧬 [summarize] folded {len(to_fold)} message(s) into a {len(new_summary)}-char summary and removed them "
          f"(summarizer prompt: {result.usage_metadata['input_tokens']} tokens)")
    return {"summary": new_summary, "buffer": [RemoveMessage(id=m.id) for m in to_fold]}


print("✅ Summarization node defined!")

In [ ]:
# ============================================================================
# CHAT NODE: Summary for Global Context, Buffer for Local Detail
# ============================================================================

def chat_llm_with_summary_node(state: SummarizationState):
    """Send the running summary and the recent buffer, not the full history."""
    messages = [SystemMessage(content=(
        "You are a helpful customer support assistant. "
        "Use the conversation summary for global context "
        "and the recent messages for local detail. "
        "Keep every answer under 60 words."
    ))]

    summary = state.get("summary", "")
    if summary:
        messages.append(SystemMessage(
            content=f"Conversation summary so far:\n{summary}"
        ))

    print(f"🤖 [chat_llm] summary: {'yes' if summary else 'no'}, buffer messages: {len(state['buffer'])}")
    messages.extend(state["buffer"])

    reply = llm.invoke(messages)

    return {"buffer": [reply]}


print("✅ Summary-aware chat node defined!")

In [ ]:
# ============================================================================
# GRAPH: Summarization Memory
# ============================================================================
# flow: START -> summarize -> chat_llm -> END

checkpointer = InMemorySaver()

builder = StateGraph(SummarizationState)
builder.add_node("summarize", summarize_buffer_node)
builder.add_node("chat_llm", chat_llm_with_summary_node)

builder.add_edge(START, "summarize")
builder.add_edge("summarize", "chat_llm")
builder.add_edge("chat_llm", END)

summary_graph = builder.compile(checkpointer=checkpointer)

print("✅ Summarization graph compiled!")

### 4.1 🧪 Facts Mentioned Early, Recalled Late

Turn 1 describes a problem. Turn 2 supplies the plan and workspace name. Turn 3 asks the
agent to recall both. By then the buffer has reached the threshold, so turns 1–2 are folded into the
summary and removed: the `[chat_llm]` line should show a summary and a single buffer message. Any correct
recall therefore came from the summary.

> **Note**: Only turn 1 seeds `summary`. Later turns pass **`buffer` only**. Because
> `summary` has no reducer, sending it again would overwrite the accumulated summary with
> whatever you passed — which is exactly how this demo silently fails.

In [ ]:
# ============================================================================
# SUMMARIZATION: Turn 1 - Seeds the Thread
# ============================================================================

config = {"configurable": {"thread_id": "ticket-summarize"}}

response1 = timed_invoke(
    summary_graph,
    {"summary": "", "buffer": [HumanMessage(content=(
        "Hi, my Team Analytics workspace isn't updating our "
        "weekly reports. The data is stuck on last Monday."
    ))]},
    config,
    channel="buffer",
)

In [ ]:
# ============================================================================
# SUMMARIZATION: Turn 2 - Supplies Details Worth Remembering
# ============================================================================
# Only `buffer` is passed - see the note above.

response2 = timed_invoke(
    summary_graph,
    {"buffer": [HumanMessage(content=(
        "We use the Growth plan with 15 seats. "
        "The workspace name is 'Marketing Performance Q4'."
    ))]},
    config,
    channel="buffer",
)

In [ ]:
# ============================================================================
# SUMMARIZATION: Turn 3 - Recall Across the Compression Boundary
# ============================================================================

response3 = timed_invoke(
    summary_graph,
    {"buffer": [HumanMessage(content=(
        "Can you remind me our workspace name "
        "and the problem we are facing?"
    ))]},
    config,
    channel="buffer",
)

In [ ]:
# ============================================================================
# INSPECT: What the Summary Actually Holds
# ============================================================================

final_state = summary_graph.get_state(config)

print("🧬 Running summary:\n")
print(final_state.values.get("summary") or "(no summary yet)")
print(f"\n📋 Messages still in buffer: {len(final_state.values.get('buffer', []))}")

---
## 📊 Part 5: Four Strategies, Measured Side by Side

Part 4 used its own conversation to test recall. To compare costs fairly, run the summarization graph on the
same six support turns as Parts 1–3 — quietly this time; only the `[summarize]` and `[chat_llm]` log lines
print — and line up the prompt tokens of every answering call.

In [ ]:
# ============================================================================
# FOUR-WAY COMPARISON: Prompt Tokens per Turn on the Same Conversation
# ============================================================================
# show=False: only the token numbers matter here, not the answers
run_conversation(summary_graph, SUPPORT_TURNS, "ticket-summary", channel="buffer", show=False)

print()
print_token_table({
    "sequential": "ticket-seq",
    "window": "ticket-sliding",
    "trim on read": "ticket-trim",
    "summary": "ticket-summary",
})


Read the columns against each other:

- **sequential** climbs on every turn — it pays for the whole thread each time
- **window** and **trim on read** level off once the window fills (turn 3 on) and track each other: same
  prompt, different storage
- **summary** grows, but more slowly than sequential: every other turn folds the buffer into the summary, so the
  prompt is the summary plus one to three recent messages — yet the summary itself gets longer with each fold
- the table counts only the answering call: each fold also pays for a summarizer call (its prompt size is on the
  `[summarize]` lines). Add those to the summary column and, on a thread this short, summarization costs about as
  much as the sequential baseline. It pays off on long threads, where replaying the full history becomes the
  expensive part — or no longer fits the context window

---
## 📝 Summary

In this notebook, we learned:

### 1. Four Strategies, One Trade-Off

| Strategy | Tokens per turn | Remembers old details? | Extra cost |
|---|---|---|---|
| **Sequential** | grows linearly, forever | ✅ everything | none |
| **Sliding window** | flat once window fills | ❌ deleted from the thread for good | none |
| **Trim on read** | flat once window fills | ❌ not in the prompt, ✅ still in storage | storage grows with the thread |
| **Summarization** | grows slowly — the summary lengthens with each fold | ✅ the gist, not the wording | one extra LLM call per fold |

### 2. The Reducer Is the Mechanism
- **`operator.add`** concatenates — it can only ever grow
- **`add_messages`** tracks message ids, which is what makes **`RemoveMessage`** work
- You cannot implement a deleting sliding window without switching reducers first
- **Trim on read** needs no reducer change: nothing is deleted, the node just sends less — and
  trimming the prompt never trims the checkpoint

### 3. Watch Out for Channels Without Reducers
- **`summary` is a plain `str`**, so any value you pass **replaces** it
- Passing `{"summary": ""}` on every turn silently wipes the memory you are trying to build —
  seed it once, then send only the buffer

### 4. Measure, Don't Assume
- Latency and token usage per turn are the only way to see these curves diverge
- Sliding window and trim on read flatten prompt size; summarization trades a second model call for a shorter prompt
- Which one is right depends on whether your conversations are *long* or merely *ongoing*

### Next Steps
- **[`01_cross_thread_memory_store.ipynb`](../../m34_long_term_agentic_memory/notebooks/01a_cross_thread_memory_store.ipynb)**
  (module 03) — memory that persists across threads, not just within one
- **[Module 11 — deep agents and harness engineering](../../m37_multi_agent_builds_deep_agents/README.md)** — context compaction at agent-harness scale
- LangChain 1.x ships **`SummarizationMiddleware`**, which applies Part 4's idea automatically
  to any `create_agent()` agent